## Optuna Hyperparameter Search — Multi-Task PCL Detection

Search for optimal hyperparameters on clean data (no dev leakage).

**Key design:**
- **Data filtering:** Only official train split par_ids (dev set excluded)
- **Focal loss with per-class alpha:** Handles imbalance — no separate class weights
- **MAX_LENGTH=192:** Covers ~99% of texts without excessive padding
- **Architecture:** SpanModel + MultiTaskTrainer from `pcl_tf.span_tf`

Results saved to `best_hyperparams.json` for `BestModel.ipynb` to consume.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    TrainingArguments,
    EarlyStoppingCallback,
)
from pcl_tf.span_tf import SpanModel, MultiTaskTrainer
from pcl_tf.dataset_manager import SpanDS, SpanTaskCollator

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

In [2]:
# --- CONFIGURATION ---
MODEL_CHECKPOINT = "albert/albert-large-v2"
CACHE_DIR = "./models_cache"
MAX_LENGTH = 192                    # reduced from 256 to fit in VRAM (p95 ≈ 128, so 192 covers ~99%)
NUM_CATEGORIES = 7
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


### Data Loading
Load binary labels and category annotations. **Critical:** Filter to official train split par_ids only to prevent data leakage.

In [3]:
# 1. Binary + text data
df = pd.read_csv("data/dontpatronizeme_pcl_cleaned.csv")
df = df.dropna(subset=["text", "par_id"])
df["par_id"] = df["par_id"].astype(int)

# 2. Category annotations → per-paragraph multi-label vectors
cats_df = pd.read_csv(
    "data/dontpatronizeme_categories.tsv",
    sep="\t", header=None, skiprows=4, engine="python",
    names=["par_id", "art_id", "text", "keyword", "country_code",
           "span_start", "span_finish", "span_text", "pcl_category",
           "num_annotators"],
)
cats_df["par_id"] = cats_df["par_id"].astype(int)

CATEGORIES = sorted(cats_df["pcl_category"].unique())
print(f"PCL categories ({len(CATEGORIES)}): {CATEGORIES}")

cat_dummies = pd.get_dummies(cats_df[["par_id", "pcl_category"]], columns=["pcl_category"], prefix="", prefix_sep="")
cat_vectors = cat_dummies.groupby("par_id")[CATEGORIES].max().astype(int)
cat_vectors = cat_vectors.reset_index()
cat_vectors["multi_label"] = cat_vectors[CATEGORIES].values.tolist()

merged = pd.merge(df, cat_vectors[["par_id", "multi_label"]], on="par_id", how="left")
merged["multi_label"] = merged["multi_label"].apply(lambda x: x if isinstance(x, list) else [0] * len(CATEGORIES))
merged["pcl_binary"] = merged["pcl_binary"].astype(int)

# --- CRITICAL: Filter to ONLY official train split par_ids ---
# pcl_cleaned.csv contains ALL data (train + dev). Excluding dev par_ids prevents leakage.
train_semeval = pd.read_csv("data/train_semeval_parids-labels.csv")
train_par_ids = set(train_semeval["par_id"].astype(int))

before = len(merged)
merged = merged[merged["par_id"].isin(train_par_ids)].reset_index(drop=True)

print(f"\nData leakage prevention: removed {before - len(merged)} dev samples")
print(f"Clean training pool: {len(merged)} samples")
print(f"  PCL positive: {merged['pcl_binary'].sum()}  |  negative: {(merged['pcl_binary'] == 0).sum()}")

PCL categories (7): ['Authority_voice', 'Compassion', 'Metaphors', 'Presupposition', 'Shallow_solution', 'The_poorer_the_merrier', 'Unbalanced_power_relations']

Data leakage prevention: removed 2093 dev samples
Clean training pool: 8375 samples
  PCL positive: 794  |  negative: 7581


In [4]:
# --- DATASET & TOKENIZER ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, cache_dir=CACHE_DIR)

train_idx, val_idx = train_test_split(
    range(len(merged)), test_size=0.1, random_state=42, stratify=merged["pcl_binary"]
)

texts = merged["text"].tolist()
binary = merged["pcl_binary"].tolist()
cats = merged["multi_label"].tolist()

def _select(idxs):
    return [texts[i] for i in idxs], [binary[i] for i in idxs], [cats[i] for i in idxs]

tr_texts, tr_bin, tr_cat = _select(train_idx)
va_texts, va_bin, va_cat = _select(val_idx)

print(f"Pre-tokenizing {len(tr_texts)} train + {len(va_texts)} val texts...")
train_ds = SpanDS(tr_texts, tr_bin, tr_cat, tokenizer, MAX_LENGTH)
val_ds = SpanDS(va_texts, va_bin, va_cat, tokenizer, MAX_LENGTH)

class_counts = np.bincount(tr_bin)

lengths = [len(ids) for ids in train_ds.input_ids]
print(f"\nToken length stats: mean={np.mean(lengths):.0f}, median={np.median(lengths):.0f}, "
      f"max={max(lengths)}, p95={np.percentile(lengths, 95):.0f}")
print(f"Train/val: {len(train_ds)}/{len(val_ds)}")
print(f"Class balance — neg: {class_counts[0]}, pos: {class_counts[1]}, ratio: {class_counts[0]/class_counts[1]:.1f}:1")
print(f"Imbalance handled via focal loss with per-class alpha (no class weights)")

Pre-tokenizing 7537 train + 838 val texts...

Token length stats: mean=62, median=55, max=192, p95=130
Train/val: 7537/838
Class balance — neg: 6822, pos: 715, ratio: 9.5:1
Imbalance handled via focal loss with per-class alpha (no class weights)


In [5]:
# --- METRICS ---
def compute_metrics_binary(pred):
    """Binary F1 for the positive (PCL) class."""
    preds = pred.predictions.argmax(-1)
    labels = pred.label_ids
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", pos_label=1, zero_division=0
    )
    return {"f1": f1, "precision": p, "recall": r, "accuracy": accuracy_score(labels, preds)}

# Pre-cache encoder config + weights in CPU RAM (avoids disk I/O per Optuna trial)
_encoder_config = AutoConfig.from_pretrained(MODEL_CHECKPOINT, cache_dir=CACHE_DIR)
_encoder_init_weights = AutoModel.from_pretrained(MODEL_CHECKPOINT, cache_dir=CACHE_DIR).cpu().state_dict()
print(f"Encoder weights cached ({sum(v.numel() * v.element_size() for v in _encoder_init_weights.values()) / 1e6:.1f} MB)")

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertModel LOAD REPORT from: albert/albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoder weights cached (70.7 MB)


### Optuna Hyperparameter Search

Search space:
- Learning rate, dropout, weight decay, warmup ratio, batch size
- **Focal alpha (0.65–0.90):** per-class weight for positives; negatives get (1 − α)
- **Focal gamma (1.0–4.0):** down-weights easy examples; higher = more focus on hard cases
- Auxiliary loss weight for category head

In [6]:
import optuna, gc, json, shutil, os, time
from optuna.pruners import MedianPruner

# Reload span_tf module to pick up any changes
import importlib, pcl_tf.span_tf
importlib.reload(pcl_tf.span_tf)
from pcl_tf.span_tf import SpanModel, MultiTaskTrainer

# Batch 8 × grad_accum 8 = effective 64. Halves peak VRAM vs batch 16.
# ALBERT doesn't support gradient checkpointing (parameter sharing), so
# smaller micro-batch is the main lever for VRAM.
BATCH_SIZE = 8
GRAD_ACCUM = 8

def _cleanup_trial(trial_model, trial_trainer, trial_number):
    """Free all GPU memory from a trial."""
    try:
        if hasattr(trial_trainer, 'optimizer') and trial_trainer.optimizer is not None:
            trial_trainer.optimizer.zero_grad(set_to_none=True)
            del trial_trainer.optimizer
        if hasattr(trial_trainer, 'lr_scheduler'):
            del trial_trainer.lr_scheduler
        if hasattr(trial_trainer, 'accelerator'):
            trial_trainer.accelerator.free_memory()
    except Exception:
        pass
    trial_model.cpu()
    del trial_model, trial_trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    trial_dir = f"./results_optuna/trial_{trial_number}"
    if os.path.exists(trial_dir):
        shutil.rmtree(trial_dir, ignore_errors=True)

def _make_fresh_encoder():
    """Build encoder from cached config + weights (no disk I/O)."""
    encoder = AutoModel.from_config(_encoder_config)
    encoder.load_state_dict(_encoder_init_weights)
    return encoder

def objective(trial):
    # --- Search space ---
    lr           = trial.suggest_float("lr", 5e-6, 5e-5, log=True)
    focal_alpha  = trial.suggest_float("focal_alpha", 0.65, 0.90)
    focal_gamma  = trial.suggest_float("focal_gamma", 1.0, 4.0)
    aux_weight   = trial.suggest_float("aux_weight", 0.05, 0.5)
    dropout      = trial.suggest_float("dropout", 0.1, 0.3)
    weight_decay = trial.suggest_float("weight_decay", 0.001, 0.1, log=True)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.05, 0.2)

    trial_model = SpanModel(
        encoder=_make_fresh_encoder(),
        num_categories=NUM_CATEGORIES,
        dropout=dropout,
    ).to(DEVICE)

    trial_collator = SpanTaskCollator(tokenizer, padding="longest")

    class OptunaPruneCallback(EarlyStoppingCallback):
        def __init__(self, trial, patience):
            super().__init__(early_stopping_patience=patience)
            self.trial = trial

        def on_evaluate(self, args, state, control, metrics=None, **kwargs):
            super().on_evaluate(args, state, control, metrics=metrics, **kwargs)
            f1 = metrics.get("eval_f1", 0.0)
            self.trial.report(f1, step=int(state.epoch))
            if self.trial.should_prune():
                raise optuna.TrialPruned()

    trial_trainer = MultiTaskTrainer(
        focal_alpha=focal_alpha,
        focal_gamma=focal_gamma,
        aux_weight=aux_weight,
        weighted_sampler=None,
        model=trial_model,
        args=TrainingArguments(
            output_dir=f"./results_optuna/trial_{trial.number}",
            eval_strategy="epoch",
            save_strategy="no",
            load_best_model_at_end=False,
            learning_rate=lr,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE * 2,
            gradient_accumulation_steps=GRAD_ACCUM,
            num_train_epochs=12,
            weight_decay=weight_decay,
            warmup_ratio=warmup_ratio,
            lr_scheduler_type="cosine",
            metric_for_best_model="f1",
            greater_is_better=True,
            label_names=["labels", "cat_labels"],
            remove_unused_columns=False,
            logging_strategy="epoch",
            fp16=True,
            torch_empty_cache_steps=50,
            dataloader_num_workers=2,
            dataloader_pin_memory=True,
            dataloader_persistent_workers=False,
            optim="adamw_torch_fused",
        ),
        data_collator=trial_collator,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics_binary,
        callbacks=[OptunaPruneCallback(trial, patience=3)],
    )

    try:
        trial_trainer.train()
        eval_f1s = [log["eval_f1"] for log in trial_trainer.state.log_history if "eval_f1" in log]
        best_f1 = max(eval_f1s) if eval_f1s else 0.0
    except Exception as e:
        best_f1 = 0.0
        raise
    finally:
        _cleanup_trial(trial_model, trial_trainer, trial.number)

    return best_f1

In [7]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Clean up any leftover GPU memory
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, "
      f"{torch.cuda.memory_reserved()/1e9:.2f} GB reserved")

study = optuna.create_study(
    direction="maximize",
    study_name="pcl_multitask_clean",
    pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=2),
)

[I 2026-03-04 01:42:02,064] A new study created in memory with name: pcl_multitask_clean


VRAM: 0.00 GB allocated, 0.00 GB reserved


In [8]:
print("Starting Optuna search (40 trials, max 12 epochs each, patience=3)...")
study.optimize(objective, n_trials=20, show_progress_bar=True)                                                   

Starting Optuna search (40 trials, max 12 epochs each, patience=3)...


  0%|          | 0/20 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
1,1.023500,0.085666,0.302128,0.181586,0.898734,0.608592
2,0.582848,0.066211,0.470588,0.384000,0.607595,0.871122
3,0.462301,0.064557,0.422145,0.290476,0.772152,0.800716
4,0.370335,0.062397,0.520548,0.407143,0.721519,0.874702
5,0.266279,0.080582,0.477987,0.475000,0.481013,0.900955
6,0.189425,0.085879,0.457143,0.416667,0.506329,0.886635
7,0.143694,0.103422,0.503401,0.544118,0.468354,0.912888


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[I 2026-03-04 04:32:49,843] Trial 18 pruned. 


Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
1,0.666485,0.044971,0.000000,0.000000,0.000000,0.904535


[I 2026-03-04 04:37:17,596] Trial 19 pruned. 


In [9]:
# --- Results ---
print(f"\n{'='*60}")
print(f"Best trial: #{study.best_trial.number}")
print(f"  Best F1: {study.best_value:.4f}")
print(f"  Params:")
for k, v in study.best_params.items():
    print(f"    {k}: {v}")

results_df = study.trials_dataframe()
results_df.to_csv("optuna_span_results.csv", index=False)
with open("best_hyperparams.json", "w") as f:
    json.dump({"best_f1": study.best_value, **study.best_params}, f, indent=2)
print(f"\nResults saved to optuna_span_results.csv")
print(f"Best hyperparams saved to best_hyperparams.json")


Best trial: #2
  Best F1: 0.5509
  Params:
    lr: 4.220013592611256e-05
    focal_alpha: 0.7564673969807376
    focal_gamma: 1.7559525538896228
    aux_weight: 0.05416432353969669
    dropout: 0.2555105754207849
    weight_decay: 0.08876598072536068
    warmup_ratio: 0.17865423948100767

Results saved to optuna_span_results.csv
Best hyperparams saved to best_hyperparams.json


### Retrain with Best Hyperparams
Train the final model using the best hyperparameters found by Optuna, with threshold optimization.

In [10]:
bp = study.best_params
print(f"Retraining with best hyperparams (F1={study.best_value:.4f})...")
print(json.dumps(bp, indent=2))

final_model = SpanModel(
    checkpoint=MODEL_CHECKPOINT,
    num_categories=NUM_CATEGORIES,
    dropout=bp["dropout"],
    cache_dir=CACHE_DIR,
).to(DEVICE)

final_model = torch.compile(final_model, dynamic=True)

final_trainer = MultiTaskTrainer(
    focal_alpha=bp["focal_alpha"],
    focal_gamma=bp["focal_gamma"],
    aux_weight=bp["aux_weight"],
    weighted_sampler=None,
    model=final_model,
    args=TrainingArguments(
        output_dir="./results_multitask",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=3,
        learning_rate=bp["lr"],
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=12,
        weight_decay=bp["weight_decay"],
        warmup_ratio=bp["warmup_ratio"],
        lr_scheduler_type="cosine",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        label_names=["labels", "cat_labels"],
        remove_unused_columns=False,
        logging_strategy="steps",
        logging_steps=50,
        fp16=True,
        torch_compile=False,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        dataloader_persistent_workers=True,
        optim="adamw_torch_fused",
    ),
    data_collator=SpanTaskCollator(tokenizer, padding="longest"),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics_binary,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

Retraining with best hyperparams (F1=0.5509)...
{
  "lr": 4.220013592611256e-05,
  "focal_alpha": 0.7564673969807376,
  "focal_gamma": 1.7559525538896228,
  "aux_weight": 0.05416432353969669,
  "dropout": 0.2555105754207849,
  "weight_decay": 0.08876598072536068,
  "warmup_ratio": 0.17865423948100767
}


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertModel LOAD REPORT from: albert/albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [11]:
print(f"Training final model (12 epochs max, patience=3)...")
final_trainer.train()

Training final model (12 epochs max, patience=3)...


W0304 04:39:26.900000 12873 torch/_inductor/utils.py:1679] [0/0_1] Not enough SMs to use max_autotune_gemm mode


Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
1,0.424218,0.055265,0.351485,0.218462,0.898734,0.687351
2,0.361124,0.037022,0.519337,0.460784,0.594937,0.896181
3,0.347490,0.038270,0.510067,0.542857,0.481013,0.912888
4,0.256904,0.043333,0.503817,0.360656,0.835443,0.844869
5,0.206374,0.042651,0.481818,0.375887,0.670886,0.863962


TrainOutput(global_step=590, training_loss=0.33357651193263166, metrics={'train_runtime': 539.1543, 'train_samples_per_second': 167.752, 'train_steps_per_second': 2.626, 'total_flos': 0.0, 'train_loss': 0.33357651193263166, 'epoch': 5.0})

In [12]:
# --- Threshold Optimization ---
preds_output = final_trainer.predict(val_ds)
raw_logits = torch.tensor(preds_output.predictions)
probs = torch.softmax(raw_logits, dim=-1)[:, 1].numpy()
true_labels = preds_output.label_ids

best_f1, best_thresh = 0, 0.5
for t in np.arange(0.20, 0.80, 0.01):
    preds_t = (probs >= t).astype(int)
    _, _, f1_t, _ = precision_recall_fscore_support(true_labels, preds_t, average="binary", pos_label=1, zero_division=0)
    if f1_t > best_f1:
        best_f1, best_thresh = f1_t, t

final_preds = (probs >= best_thresh).astype(int)
p, r, f1, _ = precision_recall_fscore_support(true_labels, final_preds, average="binary", pos_label=1, zero_division=0)
acc = accuracy_score(true_labels, final_preds)

default_preds = (probs >= 0.5).astype(int)
_, _, f1_default, _ = precision_recall_fscore_support(true_labels, default_preds, average="binary", pos_label=1, zero_division=0)

print(f"\n  Default threshold (0.50):  F1={f1_default:.4f}")
print(f"  Optimal threshold ({best_thresh:.2f}):  F1={f1:.4f}  P={p:.4f}  R={r:.4f}  Acc={acc:.4f}")
gain = (f1 - f1_default) * 100
print(f"  Threshold tuning: {'+' if gain > 0 else ''}{gain:.2f} F1 pts")

# --- Save model ---
orig_model = final_model._orig_mod if hasattr(final_model, '_orig_mod') else final_model
orig_model.encoder.save_pretrained("./pcl_multitask_model")
tokenizer.save_pretrained("./pcl_multitask_model")
torch.save({
    "model_state_dict": orig_model.state_dict(),
    "optimal_threshold": best_thresh,
    "best_f1": best_f1,
    "best_hyperparams": bp,
}, "./pcl_multitask_model/full_model.pt")

with open("./pcl_multitask_model/best_hyperparams.json", "w") as f:
    json.dump({"best_f1": float(best_f1), "optimal_threshold": float(best_thresh), **bp}, f, indent=2)

print(f"\nFinal model saved to ./pcl_multitask_model/")


  Default threshold (0.50):  F1=0.5193
  Optimal threshold (0.51):  F1=0.5227  P=0.4742  R=0.5823  Acc=0.8998
  Threshold tuning: +0.34 F1 pts


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Final model saved to ./pcl_multitask_model/
